# Analysis of precancerous tissue from BRCA1 mutation carriers

Faithful reproduction of **Figure 4A–C** and **Appendix Figure S1A–C** from Pal et al. (2021), following the authors’ [`NormBRCA1.R`](https://github.com/yunshun/HumanBreast10X/blob/main/RCode/NormBRCA1.R).

| Step | Paper setting |
|---|---|
| Samples | 8 Normal premenopausal (Total) + 4 BRCA1 pre-neoplastic (Total) |
| Integration | Seurat CCA `FindIntegrationAnchors` / `IntegrateData` |
| Dims | 30 PCs; t-SNE `seed.use = 2018` |
| All-cell clustering | Louvain, resolution **0.12** |
| Non-epithelial clustering | resolution **0.08**; drop epithelial clusters **2, 4, 5, 8** |
| Markers | edgeR quasi-likelihood on patient×cluster pseudo-bulks |

**This notebook does not use the scanpy workarounds** (NumPy PCA, gauss neighbors, resolution fallback). It prepares 10X inputs from GSE161529 and runs `NormBRCA1_faithful.R`.

**Recommended environment:** Linux (e.g. SageMaker `ml.r5.2xlarge`, 64 GB RAM), R ≥ 4.x with Seurat, edgeR, limma, pheatmap. GPU not required.

## 0. Initializations

In [ ]:
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from signals_in_the_noise.config import get_data_path

NOTEBOOK_DIR = Path.cwd()
# Resolve repo paths whether the kernel cwd is the notebook folder or the repo root.
for candidate in (NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent, NOTEBOOK_DIR.parent.parent.parent):
    if (candidate / "notebooks" / "GSE161529" / "necb" / "NormBRCA1_faithful.R").exists():
        REPO_ROOT = candidate
        break
    if (candidate / "NormBRCA1_faithful.R").exists():
        REPO_ROOT = candidate.parent.parent.parent  # necb → GSE161529 → notebooks → repo
        break
else:
    REPO_ROOT = NOTEBOOK_DIR

R_SCRIPT = REPO_ROOT / "notebooks" / "GSE161529" / "necb" / "NormBRCA1_faithful.R"
if not R_SCRIPT.exists():
    R_SCRIPT = Path("NormBRCA1_faithful.R").resolve()

RAW_DIR = get_data_path("raw") / "GSE161529_RAW"
FEATURES = get_data_path("raw") / "GSE161529_features.tsv.gz"
ANNOTATIONS = get_data_path("GSE161529_annotations_df.csv")

WORK_DIR = get_data_path("GSE161529_analysis_cache") / "brca1_preneoplastic_seurat"
DATA_ROOT = WORK_DIR / "Data"
OUT_DIR = WORK_DIR / "output"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("R_SCRIPT :", R_SCRIPT, R_SCRIPT.exists())
print("RAW_DIR  :", RAW_DIR, RAW_DIR.exists())
print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR  :", OUT_DIR)

In [ ]:
# Author sample IDs → GEO barcode/matrix stems in this repo
SAMPLE_MAP = {
    "N-0019-total": "GSM4909254_N-PM0019-Total",
    "N-0233-total": "GSM4909265_N-PM0233-Total",
    "N-0092-total": "GSM4909253_N-PM0092-Total",
    "N-0230.17-total": "GSM4909261_N-PM0230-Total",
    "N-0093-total": "GSM4909257_N-PM0095-Total",
    "N-0123-total": "GSM4909268_N-MH0023-Total",
    "N-0064-total": "GSM4909263_N-MH0064-Total",
    "N-0169-total": "GSM4909266_N-MH0169-Total",
    "B1-0894": "GSM4909277_B1-KCF0894",
    "B1-0033": "GSM4909278_B1-MH0033",
    "B1-0023": "GSM4909279_B1-MH0023",
    "B1-0090": "GSM4909280_B1-MH0090",
}

assert ANNOTATIONS.exists(), f"Missing annotations: {ANNOTATIONS}"
assert FEATURES.exists(), f"Missing features: {FEATURES}"
assert RAW_DIR.exists(), f"Missing RAW dir: {RAW_DIR}"

ann = pd.read_csv(ANNOTATIONS)
ann_by_name = ann.set_index("sample-name")
missing = [s for s in SAMPLE_MAP if s not in ann_by_name.index]
assert not missing, f"Samples missing from annotations: {missing}"
ann_by_name.loc[list(SAMPLE_MAP)].filter(
    items=["mito-upper", "genes-lower", "genes-upper", "library-size-upper", "number-of-cells-after-filtering"]
)

## 1. Prepare author-style 10X directories + SampleStats.txt

Creates one folder per sample with `barcodes.tsv.gz`, `features.tsv.gz`, `matrix.mtx.gz`, and writes QC thresholds from the paper’s supplementary tables (already in `GSE161529_annotations_df.csv`).

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


stats_rows = []
for sample_id, stem in SAMPLE_MAP.items():
    sample_dir = DATA_ROOT / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)

    barcodes = RAW_DIR / f"{stem}-barcodes.tsv.gz"
    matrix = RAW_DIR / f"{stem}-matrix.mtx.gz"
    if not barcodes.exists() or not matrix.exists():
        raise FileNotFoundError(f"Missing 10X files for {sample_id}: {barcodes.name}, {matrix.name}")

    link_or_copy(barcodes, sample_dir / "barcodes.tsv.gz")
    link_or_copy(matrix, sample_dir / "matrix.mtx.gz")
    link_or_copy(FEATURES, sample_dir / "features.tsv.gz")

    row = ann_by_name.loc[sample_id]
    stats_rows.append(
        {
            "SampleName": sample_id,
            "Mito": float(row["mito-upper"]),
            "GeneLower": int(row["genes-lower"]),
            "GeneUpper": int(row["genes-upper"]),
            "LibSize": int(row["library-size-upper"]),
        }
    )
    print(f"Prepared {sample_id}")

stats_df = pd.DataFrame(stats_rows)
stats_path = DATA_ROOT / "SampleStats.txt"
stats_df.to_csv(stats_path, sep="\t", index=False)
display(stats_df)
print("Wrote", stats_path)

### Optional: NCBI gene-info file (author symbol remap)

Authors used `limma::alias2SymbolUsingNCBI` with `180808_Homo_sapiens.gene_info.gz`. If absent, the R script keeps Cell Ranger symbols as `Official` (usually fine for this GEO release).

In [ ]:
gene_info = DATA_ROOT / "180808_Homo_sapiens.gene_info.gz"
if gene_info.exists():
    print("Found", gene_info)
else:
    print(
        "Optional file not present:\n",
        gene_info,
        "\nPlace the NCBI gene_info gzip here for exact author symbol remapping.",
    )

## 2. Run faithful Seurat + edgeR pipeline

Invokes `NormBRCA1_faithful.R` (same integration / clustering / pseudo-bulk parameters as the paper).

The next cell finds absolute `Rscript` and installs **Seurat**, **edgeR**, **limma**, **ggplot2**, **pheatmap** into that R (conda-forge first, then BiocManager).


In [ ]:
import os
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

# --- locate Rscript (same logic as before) ---
def find_rscript() -> Path | None:
    candidates: list[Path] = []
    py_bin = Path(sys.executable).resolve().parent
    candidates.append(py_bin / "Rscript")
    candidates.append(py_bin.parent / "bin" / "Rscript")
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates.append(Path(conda_prefix) / "bin" / "Rscript")
    which = shutil.which("Rscript")
    if which:
        candidates.append(Path(which).resolve())
    home = Path.home()
    for pattern in (
        "anaconda3/envs/*/bin/Rscript",
        "miniconda3/envs/*/bin/Rscript",
        "SageMaker/.conda/envs/*/bin/Rscript",
        ".conda/envs/*/bin/Rscript",
    ):
        candidates.extend(home.glob(pattern))
    candidates.extend(
        [
            Path("/opt/conda/bin/Rscript"),
            Path("/opt/conda/envs/signals-in-the-noise/bin/Rscript"),
            Path("/usr/bin/Rscript"),
            Path("/usr/local/bin/Rscript"),
        ]
    )
    seen: set[str] = set()
    for c in candidates:
        try:
            c = Path(c).expanduser().resolve()
        except OSError:
            continue
        if str(c) in seen:
            continue
        seen.add(str(c))
        if c.is_absolute() and c.is_file():
            return c
    return None


def find_conda() -> str | None:
    conda = shutil.which("conda")
    if conda:
        return conda
    for cand in (
        Path(sys.executable).resolve().parent / "conda",
        Path("/opt/conda/bin/conda"),
        Path.home() / "anaconda3" / "bin" / "conda",
        Path.home() / "miniconda3" / "bin" / "conda",
    ):
        if cand.is_file():
            return str(cand)
    return None


def install_r_into_kernel_env() -> Path | None:
    prefix = Path(sys.executable).resolve().parent.parent
    conda = find_conda()
    if conda is None:
        print("conda not found; cannot auto-install r-base")
        return None
    cmd = [conda, "install", "-y", "-c", "conda-forge", "-p", str(prefix), "r-base"]
    print("Running:", " ".join(cmd))
    print("conda exit:", subprocess.run(cmd, check=False).returncode)
    return find_rscript()


REQUIRED_R_PKGS = ("Seurat", "edgeR", "limma", "ggplot2", "pheatmap")
# conda-forge names for the same packages
CONDA_R_PKGS = (
    "r-seurat",
    "bioconductor-edger",
    "bioconductor-limma",
    "r-ggplot2",
    "r-pheatmap",
)


def missing_r_packages(rscript: Path) -> list[str]:
    missing: list[str] = []
    for pkg in REQUIRED_R_PKGS:
        code = f'if (!requireNamespace("{pkg}", quietly=TRUE)) quit(status=2)'
        rc = subprocess.run(
            [str(rscript), "-e", code],
            check=False,
            capture_output=True,
            text=True,
        ).returncode
        if rc != 0:
            missing.append(pkg)
    return missing


def install_r_packages(rscript: Path) -> None:
    missing = missing_r_packages(rscript)
    if not missing:
        print("R packages OK:", ", ".join(REQUIRED_R_PKGS))
        return

    print("Missing R packages:", ", ".join(missing))
    # Prefer conda into the env that owns this Rscript
    prefix = rscript.resolve().parent.parent
    conda = find_conda()
    if conda is not None:
        cmd = [
            conda,
            "install",
            "-y",
            "-c",
            "conda-forge",
            "-p",
            str(prefix),
            *CONDA_R_PKGS,
        ]
        print("Installing via conda (may take several minutes):\n ", " ".join(cmd))
        rc = subprocess.run(cmd, check=False).returncode
        print("conda exit:", rc)
        missing = missing_r_packages(rscript)
        if not missing:
            print("R packages OK after conda install")
            return
        print("Still missing after conda:", ", ".join(missing))

    # Fallback: BiocManager + install.packages into this R
    r_code = textwrap.dedent(
        """
        options(repos = c(CRAN = "https://cloud.r-project.org"))
        pkgs <- c("Seurat", "ggplot2", "pheatmap")
        bioc <- c("edgeR", "limma")
        for (p in pkgs) {
          if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
        }
        if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
        for (p in bioc) {
          if (!requireNamespace(p, quietly = TRUE)) BiocManager::install(p, update = FALSE, ask = FALSE)
        }
        """
    )
    print("Falling back to install.packages / BiocManager...")
    rc = subprocess.run([str(rscript), "-e", r_code], check=False).returncode
    print("R install exit:", rc)
    missing = missing_r_packages(rscript)
    if missing:
        raise RuntimeError(
            "Still missing R packages: "
            + ", ".join(missing)
            + f"\nInstall into the env that owns {rscript}:\n"
            f"  conda install -y -c conda-forge -p {prefix} "
            + " ".join(CONDA_R_PKGS)
        )
    print("R packages OK after R install")


print("sys.executable:", sys.executable)
print("kernel env prefix:", Path(sys.executable).resolve().parent.parent)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))

rscript = find_rscript()
if rscript is None:
    print("Rscript not found — attempting conda install into THIS kernel's env...")
    rscript = install_r_into_kernel_env()

if rscript is None or not Path(rscript).is_absolute() or not Path(rscript).is_file():
    prefix = Path(sys.executable).resolve().parent.parent
    raise RuntimeError(
        "Could not resolve an absolute Rscript path for this kernel.\n"
        f"  conda install -y -c conda-forge -p {prefix} r-base"
    )

rscript = Path(rscript).resolve()
print("Using absolute Rscript:", rscript)
subprocess.run([str(rscript), "--version"], check=True)
install_r_packages(rscript)
print("OK — ready to run NormBRCA1_faithful.R")


In [ ]:
import subprocess
from pathlib import Path

assert R_SCRIPT.exists(), f"Missing R script: {R_SCRIPT}"

if "rscript" not in dir() or rscript is None:
    raise RuntimeError("Set rscript to an absolute Path first (run the previous cell).")

rscript = Path(rscript).expanduser().resolve()
if not rscript.is_absolute() or not rscript.is_file():
    raise FileNotFoundError(
        f"rscript must be an absolute existing file, got: {rscript!r}."
    )

cmd = [str(rscript), str(R_SCRIPT), str(DATA_ROOT), str(OUT_DIR)]
print("Running:\n ", " ".join(cmd))
completed = subprocess.run(cmd, check=False)
print("exit code:", completed.returncode)
if completed.returncode != 0:
    raise RuntimeError(
        f"NormBRCA1_faithful.R failed with exit code {completed.returncode}. "
        "Check the R console output above."
    )


## 3. Display paper figures

Outputs written under `OUT_DIR`:

| File | Panel |
|---|---|
| `Fig4A.pdf` | t-SNE by sample |
| `Fig4B.pdf` | t-SNE by cluster (res 0.12) |
| `Fig4C.pdf` | Non-epithelial recluster (res 0.08) |
| `FigS1A-left.pdf` / `FigS1A-right.pdf` | Condition split + proportions |
| `FigS1C.pdf` | Pseudo-bulk marker heatmap |
| `SeuratObject_NormB1Total.rds` | Integrated object |
| `cluster_by_condition_counts.csv` | Counts for abundance testing |

In [ ]:
outputs = sorted(OUT_DIR.glob("*"))
for p in outputs:
    print(f"{p.name:40s} {p.stat().st_size/1e6:8.2f} MB")

In [ ]:
# Convert PDFs to PNG for inline display when ImageMagick/pdftoppm is available;
# otherwise open the PDFs from OUT_DIR directly.
pdfs = ["Fig4A.pdf", "Fig4B.pdf", "Fig4C.pdf", "FigS1A-right.pdf", "FigS1C.pdf"]
for name in pdfs:
    pdf = OUT_DIR / name
    if not pdf.exists():
        print("missing", name)
        continue
    png = OUT_DIR / (pdf.stem + ".png")
    if not png.exists():
        converter = shutil.which("pdftoppm") or shutil.which("magick")
        if converter and "pdftoppm" in converter:
            subprocess.run([converter, "-png", "-singlefile", "-r", "120", str(pdf), str(OUT_DIR / pdf.stem)], check=False)
        elif converter:
            subprocess.run([converter, "convert", "-density", "120", str(pdf), str(png)], check=False)
    if png.exists():
        print(name)
        display(Image(filename=str(png)))
    else:
        print(f"{name} ready at {pdf}")

## 4. Sanity checks vs paper

| Checkpoint | Expected |
|---|---|
| Cells after QC | 59,766 (23,240 BRCA1 + 36,526 Normal) |
| Integration | Seurat CCA, 30 PCs |
| Cluster resolution | 0.12 (all), 0.08 (stromal) |
| Epithelial drop | clusters 2, 4, 5, 8 |
| Composition Normal vs BRCA1 | *P* ≈ 0.14 (quasi-multinomial; see author `NormTotal.R` / Appendix S1B) |
| Stromal DE | No striking WT vs BRCA1 differences (Appendix S1C) |

In [ ]:
counts_path = OUT_DIR / "cluster_by_condition_counts.csv"
if counts_path.exists():
    counts = pd.read_csv(counts_path, index_col=0)
    display(counts)
    print("Total cells in abundance table:", int(counts.to_numpy().sum()))
else:
    print("Run the R pipeline first to produce cluster_by_condition_counts.csv")